# Pipeline MLOps – Predicción de Stroke
**Materia:** MLOps1 – CEIA – FIUBA  
**Nombre:** Julia Maldonado_a2319  
**Dataset:** Healthcare Stroke Dataset  
**Modelos:** Random Forest + XGBoost (Ensamble Soft Voting)  

Este notebook implementa el ciclo completo de MLOps en modo **modo local para prototipado — ver docker-compose.yml para ejecución en producción**:
1. Carga y preprocesamiento de datos
2. Búsqueda de hiperparámetros con tracking en MLflow
3. Registro del mejor modelo en el Model Registry de MLflow
4. Exportación del modelo para ser servido por FastAPI

Instalación de dependencias

In [ ]:
# Instalacion de dependencias necesarias si no están instaladas
%pip install mlflow scikit-learn xgboost imbalanced-learn pandas numpy joblib fastapi uvicorn python-multipart --quiet

Note: you may need to restart the kernel to use updated packages.


## 1. Importaciones y configuración de MLflow

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix
)
from xgboost import XGBClassifier
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings("ignore")

# ── Configuración de MLflow ───────────────────────────────────────────────
# Modo local: los experimentos y artefactos se guardan en la carpeta mlruns/
# dentro del directorio donde se ejecuta este notebook.
MLFLOW_TRACKING_URI = os.path.join(os.getcwd(), "mlruns")
EXPERIMENT_NAME     = "stroke_prediccion"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experimento:         {EXPERIMENT_NAME}")

MLflow tracking URI: file:///C:/Users/steph/Downloads/.MAESTRIA/proyectos_python/MLOps1/TPFINAL MLOPS/stroke_mlops/notebooks/mlruns
Experimento:         stroke_prediccion


## 2. Carga de datos

In [ ]:
# ── Ruta al CSV ────────────────────────────────────────────────────
DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), "data", "healthcare-dataset-stroke-data.csv")  

df = pd.read_csv(DATA_PATH)
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
print(f"Distribución de stroke:\n{df['stroke'].value_counts()}")
df.head()

Filas: 5110 | Columnas: 12
Distribución de stroke:
stroke
0    4861
1     249
Name: count, dtype: int64


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## 3. Preprocesamiento
Misma estrategia del trabajo original de AMq1:
- Eliminar fila con género 'Other'
- Encoding binario para `gender` y `ever_married`
- One-Hot Encoding para variables categóricas de más de 2 categorías
- Imputación de BMI con mediana estratificada por `stroke`
- Capping de outliers con IQR
- Balanceo con RandomUnderSampler

In [4]:
def preprocess(df: pd.DataFrame) -> tuple:
    """
    Aplica el preprocesamiento completo al DataFrame raw.

    Pasos:
        1. Elimina la fila con género 'Other'.
        2. Encoding binario para 'gender' y 'ever_married'.
        3. One-Hot Encoding para 'work_type', 'Residence_type', 'smoking_status'.
        4. Separa features (X) y target (y).
        5. Split train/test (80/20, random_state=42).
        6. Imputa nulos en BMI con la mediana estratificada por stroke en train.
        7. Capping de outliers en BMI con límites IQR calculados en train.
        8. Convierte booleanos a int.
        9. Elimina la columna 'id'.
        10. Balancea con RandomUnderSampler sobre el conjunto completo y re-splitea.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame original cargado desde el CSV.

    Returns
    -------
    tuple : (X_train_rs, X_test_rs, y_train_rs, y_test_rs, feature_names, bmi_stats)
        - X_train_rs, X_test_rs : DataFrames de features train/test balanceados.
        - y_train_rs, y_test_rs : Series de target train/test balanceados.
        - feature_names          : lista de columnas de features.
        - bmi_stats              : dict con medianas y límites IQR para inferencia.
    """
    df = df.copy()

    # 1. Eliminar género 'Other'
    df = df[df['gender'] != 'Other'].reset_index(drop=True)

    # 2. Encoding binario
    df['gender']       = (df['gender'] == 'Female').astype(int)
    df['ever_married'] = (df['ever_married'] == 'Yes').astype(int)

    # 3. One-Hot Encoding
    df = pd.get_dummies(df, columns=['work_type', 'Residence_type', 'smoking_status'])

    # 4. Separar X e y
    X = df.drop(columns=['stroke'])
    y = df['stroke']

    # 5. Split train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    # 6. Imputar BMI con mediana estratificada (calculada solo en train)
    median_bmi_1 = X_train.loc[y_train == 1, 'bmi'].median()
    median_bmi_0 = X_train.loc[y_train == 0, 'bmi'].median()

    X_train.loc[(X_train['bmi'].isnull()) & (y_train == 1), 'bmi'] = median_bmi_1
    X_train.loc[(X_train['bmi'].isnull()) & (y_train == 0), 'bmi'] = median_bmi_0
    X_test.loc[(X_test['bmi'].isnull())  & (y_test == 1),  'bmi'] = median_bmi_1
    X_test.loc[(X_test['bmi'].isnull())  & (y_test == 0),  'bmi'] = median_bmi_0

    # 7. Capping de outliers IQR (límites calculados solo en train)
    Q1, Q3  = X_train['bmi'].quantile(0.25), X_train['bmi'].quantile(0.75)
    IQR     = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR

    X_train['bmi'] = X_train['bmi'].clip(lim_inf, lim_sup)
    X_test['bmi']  = X_test['bmi'].clip(lim_inf, lim_sup)

    # 8. Convertir booleanos a int
    for _X in [X_train, X_test]:
        bool_cols = _X.select_dtypes(include='bool').columns
        _X[bool_cols] = _X[bool_cols].astype(int)

    # 9. Eliminar columna id
    X_train = X_train.drop(columns=['id'])
    X_test  = X_test.drop(columns=['id'])

    # Guardar nombres de columnas y stats para inferencia
    feature_names = X_train.columns.tolist()
    bmi_stats = {
        'median_bmi_1': median_bmi_1,
        'median_bmi_0': median_bmi_0,
        'lim_inf': lim_inf,
        'lim_sup': lim_sup,
    }

    # 10. Balanceo con RandomUnderSampler sobre el dataset completo
    X_full = X_train._append(X_test)
    y_full = y_train._append(y_test)

    rus = RandomUnderSampler(sampling_strategy='majority', random_state=42)
    X_under, y_under = rus.fit_resample(X_full, y_full)

    X_train_rs, X_test_rs, y_train_rs, y_test_rs = train_test_split(
        X_under, y_under, test_size=0.25, random_state=43
    )

    return X_train_rs, X_test_rs, y_train_rs, y_test_rs, feature_names, bmi_stats


X_train, X_test, y_train, y_test, feature_names, bmi_stats = preprocess(df)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Features ({len(feature_names)}): {feature_names}")
print(f"\nBMI stats guardados para inferencia: {bmi_stats}")

Train: (373, 18) | Test: (125, 18)
Features (18): ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'avg_glucose_level', 'bmi', 'work_type_Govt_job', 'work_type_Never_worked', 'work_type_Private', 'work_type_Self-employed', 'work_type_children', 'Residence_type_Rural', 'Residence_type_Urban', 'smoking_status_Unknown', 'smoking_status_formerly smoked', 'smoking_status_never smoked', 'smoking_status_smokes']

BMI stats guardados para inferencia: {'median_bmi_1': 29.7, 'median_bmi_0': 27.9, 'lim_inf': np.float64(10.300000000000006), 'lim_sup': np.float64(46.29999999999999)}


## 4. Búsqueda de hiperparámetros con MLflow

Se entrenan **tres variantes de modelos** y cada run queda registrado en MLflow:
- Random Forest con `RandomizedSearchCV`
- XGBoost con `RandomizedSearchCV`  
- Ensamble Soft Voting (RF + XGB)

In [5]:
def log_run(run_name: str, model, X_tr, X_te, y_tr, y_te, params: dict) -> str:
    """
    Entrena el modelo, calcula métricas y registra todo en un run de MLflow.

    Parameters
    ----------
    run_name : str
        Nombre del run que aparecerá en la UI de MLflow.
    model :
        Estimador sklearn ya inicializado.
    X_tr, X_te : pd.DataFrame
        Features de entrenamiento y test.
    y_tr, y_te : pd.Series
        Target de entrenamiento y test.
    params : dict
        Hiperparámetros a loguear manualmente (para CV wrappers).

    Returns
    -------
    str
        run_id del experimento registrado.
    """
    with mlflow.start_run(run_name=run_name) as run:
        # Entrenamiento
        model.fit(X_tr, y_tr)
        y_pred  = model.predict(X_te)
        y_proba = model.predict_proba(X_te)[:, 1]

        # Métricas
        metrics = {
            "accuracy" : round(accuracy_score(y_te, y_pred), 4),
            "roc_auc"  : round(roc_auc_score(y_te, y_proba), 4),
            "f1_score" : round(f1_score(y_te, y_pred), 4),
        }

        # Log a MLflow
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)

        # Guardar el artefacto del modelo
        signature = infer_signature(X_tr, model.predict(X_tr))
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            signature=signature,
            registered_model_name=f"stroke_{run_name.lower().replace(' ', '_')}"
        )

        # Loguear los nombres de las features como tag
        mlflow.set_tag("features", str(feature_names))
        mlflow.set_tag("model_type", run_name)

        print(f"[{run_name}] AUC={metrics['roc_auc']} | F1={metrics['f1_score']} | run_id={run.info.run_id}")
        return run.info.run_id

In [ ]:
# ── Random Forest con búsqueda de hiperparámetros ────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_param_grid = {
    'n_estimators'    : [90, 100, 115, 130, 200],
    'max_depth'       : list(range(2, 20)),
    'min_samples_split': list(range(2, 10)),
    'min_samples_leaf' : list(range(1, 10)),
    'criterion'       : ['entropy'],
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=rf_param_grid,
    n_iter=20,          # este valor se aumenta en caso de una búsqueda más exhaustiva
    cv=cv,
    scoring='roc_auc',
    random_state=35,
    n_jobs=-1,
    verbose=1,
)

rf_run_id = log_run(
    run_name="Random Forest",
    model=rf_search,
    X_tr=X_train, X_te=X_test,
    y_tr=y_train, y_te=y_test,
    params={"cv_folds": 5, "n_iter": 20, "scoring": "roc_auc", **rf_param_grid},
)

print(f"\nMejores params RF: {rf_search.best_params_}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits


2026/04/25 15:31:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/25 15:31:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[Random Forest] AUC=0.8326 | F1=0.803 | run_id=dc8cfafb921f43a7b533746b477f52fd

Mejores params RF: {'n_estimators': 100, 'min_samples_split': 8, 'min_samples_leaf': 6, 'max_depth': 4, 'criterion': 'entropy'}


Registered model 'stroke_random_forest' already exists. Creating a new version of this model...
Created version '2' of model 'stroke_random_forest'.


In [7]:
# ── XGBoost con búsqueda de hiperparámetros ───────────────────────────────────
xgb_param_grid = {
    'n_estimators' : [200, 300, 400],
    'max_depth'    : [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample'    : [0.7, 0.8, 0.9],
}

xgb_search = RandomizedSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42),
    param_distributions=xgb_param_grid,
    n_iter=20,
    cv=cv,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

xgb_run_id = log_run(
    run_name="XGBoost",
    model=xgb_search,
    X_tr=X_train, X_te=X_test,
    y_tr=y_train, y_te=y_test,
    params={"cv_folds": 5, "n_iter": 20, "scoring": "roc_auc", **xgb_param_grid},
)

print(f"\nMejores params XGB: {xgb_search.best_params_}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits


2026/04/25 15:32:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/25 15:32:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[XGBoost] AUC=0.849 | F1=0.7939 | run_id=1fce3d2ee50f429ab0d07ec020ca5542

Mejores params XGB: {'subsample': 0.8, 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.01}


Registered model 'stroke_xgboost' already exists. Creating a new version of this model...
Created version '2' of model 'stroke_xgboost'.


In [8]:
# ── Ensamble Soft Voting (RF + XGB, pesos iguales, sin árbol de Gini) ────────
ensemble = VotingClassifier(
    estimators=[
        ("rf",  rf_search),
        ("xgb", xgb_search),
    ],
    voting="soft",
    n_jobs=-1,
)

ensemble_run_id = log_run(
    run_name="Ensemble RF XGB",
    model=ensemble,
    X_tr=X_train, X_te=X_test,
    y_tr=y_train, y_te=y_test,
    params={"voting": "soft", "estimators": "rf, xgb"},
)

2026/04/25 15:32:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/25 15:32:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[Ensemble RF XGB] AUC=0.8433 | F1=0.812 | run_id=75b807da85db45dd9a201a02e6f38dfe


Registered model 'stroke_ensemble_rf_xgb' already exists. Creating a new version of this model...
Created version '2' of model 'stroke_ensemble_rf_xgb'.


## 5. Selección del mejor modelo y exportación

Comparamos los runs del experimento y guardamos el mejor modelo como `model.pkl` para que la API lo cargue.

In [10]:
import mlflow.tracking

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

# Obtener todos los runs del experimento, ordenados por AUC descendente
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.roc_auc DESC"]
)

print("Comparación de todos los runs:")
print(f"{'Nombre':<25} {'AUC':>8} {'F1':>8} {'Accuracy':>10}")
print("-" * 55)
for r in runs:
    print(
        f"{r.data.tags.get('model_type','?'):<25}"
        f" {r.data.metrics.get('roc_auc',0):>8.4f}"
        f" {r.data.metrics.get('f1_score',0):>8.4f}"
        f" {r.data.metrics.get('accuracy',0):>10.4f}"
    )

best_run = runs[0]
print(f"\nMejor modelo: {best_run.data.tags.get('model_type')} (run_id={best_run.info.run_id})")

Comparación de todos los runs:
Nombre                         AUC       F1   Accuracy
-------------------------------------------------------
XGBoost                     0.8490   0.7939     0.7840
XGBoost                     0.8490   0.7939     0.7840
Ensemble RF XGB             0.8433   0.8120     0.8000
Ensemble RF XGB             0.8433   0.8120     0.8000
Random Forest               0.8326   0.8030     0.7920
Random Forest               0.8326   0.8030     0.7920

Mejor modelo: XGBoost (run_id=1fce3d2ee50f429ab0d07ec020ca5542)


In [ ]:
# Cargar el mejor modelo desde MLflow y exportarlo como .pkl para la API
best_model_uri = f"runs:/{best_run.info.run_id}/model"
best_model     = mlflow.sklearn.load_model(best_model_uri)

# Carpeta de salida donde la API buscará el modelo
MODEL_OUTPUT_DIR = os.path.join(os.path.dirname(os.getcwd()), "api") 
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

# Guardar modelo
joblib.dump(best_model, os.path.join(MODEL_OUTPUT_DIR, "model.pkl"))

# Guardar stats de preprocesamiento (medianas y límites IQR)
joblib.dump(bmi_stats,     os.path.join(MODEL_OUTPUT_DIR, "bmi_stats.pkl"))
joblib.dump(feature_names, os.path.join(MODEL_OUTPUT_DIR, "feature_names.pkl"))

print(f"Modelo exportado a: {os.path.join(MODEL_OUTPUT_DIR, 'model.pkl')}")
print(f"Stats guardados en: {MODEL_OUTPUT_DIR}/bmi_stats.pkl y feature_names.pkl")

Modelo exportado a: C:/Users/steph/Downloads/.MAESTRIA/proyectos_python/MLOps1/TPFINAL MLOPS/stroke_mlops/api\model.pkl
Stats guardados en: C:/Users/steph/Downloads/.MAESTRIA/proyectos_python/MLOps1/TPFINAL MLOPS/stroke_mlops/api/bmi_stats.pkl y feature_names.pkl


## 6. Verificacion de Resultados UI MLflow/ REST API

### UI de MLflow
Para explorar los experimentos y runs registrados, se ejecuta en una terminal:

```bash
mlflow ui --backend-store-uri "file:///C:/Users/steph/Downloads/.MAESTRIA/proyectos_python/MLOps1/TPFINAL MLOPS/stroke_mlops/notebooks/mlruns" --port 5001
```

Luego se abre http://localhost:5001 en el navegador.

### API REST - Stroke Prediction
Para levantar la API se ejecuta en una terminal:

```bash
cd "C:/Users/steph/Downloads/.MAESTRIA/proyectos_python/MLOps1/TPFINAL MLOPS/stroke_mlops/api"
uvicorn main:app --reload --port 8000
```

Luego se abre en navegador http://localhost:8000/docs para ver la documentación interactiva.